In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv()

os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ["LANGCHAIN_PROJECT"] = "HelwanChem-Eval"

In [73]:
from langsmith import Client

client = Client()

# Create dataset
dataset = client.create_dataset(
    dataset_name="Helwan Chem chatbot evaluation"
)

# Create examples
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"input": "What is Helwan Chemical Industries Company?"},
            "outputs": {"output": "Helwan Chemical Industries Company is an Egyptian public sector company specializing in the production of chemical products used in industrial, agricultural, and commercial applications."}
        },
        {
            "inputs": {"input": "What products does Helwanchem manufacture?"},
            "outputs": {"output": "Helwanchem manufactures a range of chemical products including industrial chemicals, fertilizers, and chemical compounds used in manufacturing and agriculture."}
        },
        {
            "inputs": {"input": "Is Helwanchem a government-owned company?"},
            "outputs": {"output": "Yes, Helwan Chemical Industries Company is affiliated with the Egyptian public sector and operates under government ownership."}
        },
        {
            "inputs": {"input": "Where is Helwanchem located?"},
            "outputs": {"output": "Helwanchem is located in Helwan, Cairo, Egypt, which is a major industrial zone."}
        },
        {
            "inputs": {"input": "How can I contact Helwanchem?"},
            "outputs": {"output": "You can contact Helwanchem through the official website contact page, by phone, or by visiting the company headquarters in Helwan."}
        },
        {
            "inputs": {"input": "Does Helwanchem provide safety data sheets for its products?"},
            "outputs": {"output": "Yes, Helwanchem provides safety data sheets (SDS) and follows standard chemical safety regulations."}
        },
        {
            "inputs": {"input": "Which industries use Helwanchem products?"},
            "outputs": {"output": "Helwanchem products are used in industries such as agriculture, manufacturing, construction, and chemical processing."}
        },
        {
            "inputs": {"input": "Does Helwanchem export its products?"},
            "outputs": {"output": "Helwanchem serves both local and regional markets, and some products may be available for export depending on regulations and agreements."}
        },
        {
            "inputs": {"input": "How can I apply for a job at Helwanchem?"},
            "outputs": {"output": "Job applications can be submitted through official announcements on the Helwanchem website or through public sector recruitment channels."}
        },
        {
            "inputs": {"input": "Does Helwanchem comply with environmental regulations?"},
            "outputs": {"output": "Yes, Helwanchem operates in compliance with Egyptian environmental laws and applies industrial safety and environmental protection standards."}
        },
        {
            "inputs": {"input": "Who are Helwanchem’s main customers?"},
            "outputs": {"output": "Helwanchem serves industrial companies, agricultural suppliers, and government-related entities that require chemical products."}
        },
        {
            "inputs": {"input": "Does Helwanchem offer bulk chemical supply?"},
            "outputs": {"output": "Yes, Helwanchem supplies chemicals in bulk quantities based on customer requirements and contractual agreements."}
        },
        {
            "inputs": {"input": "Can individuals purchase directly from Helwanchem?"},
            "outputs": {"output": "Purchases are typically made through corporate or institutional agreements rather than individual retail sales."}
        },
        {
            "inputs": {"input": "Is Helwanchem involved in fertilizer production?"},
            "outputs": {"output": "Yes, fertilizer-related chemical products are part of Helwanchem’s production portfolio."}
        },
        {
            "inputs": {"input": "How does Helwanchem ensure product quality?"},
            "outputs": {"output": "Helwanchem follows industrial quality control procedures and internal testing standards to ensure product consistency and safety."}
        }
    ]
)

{'example_ids': ['b81a6cb1-73cc-46e2-85df-d83682d5421e',
  '51adaf22-072d-4863-8450-f5e72ae8e9f6',
  'a9bc1004-8708-44f7-a940-9752ec96d13f',
  'dbd1e2f9-71bb-4a63-8c0a-949eb0d4e8ca',
  'a254eff1-e17c-4631-b275-634f58cfa6c0',
  '7076596e-c563-42b9-b2b6-5601676ae134',
  '0b748643-9a5d-453a-b208-9e95927b88da',
  '23002e75-01f1-4509-bcd9-fad7839fd36c',
  '714afad8-dd78-488b-bae8-423b060f7948',
  'e8f1b3e5-31e7-49eb-8054-1c1cbfd92cb3',
  'f506640a-dd02-43c3-9ad0-9be44d5cbb16',
  '6dcd5bbe-d083-400f-93f2-705e0e996cd1',
  '03701d07-f193-44c8-b314-5505484a9dd0',
  '1b6fbd85-d573-451b-9cef-c93b97bfb083',
  '253e9bfa-5eef-497f-b16d-eed0f82640bd'],
 'count': 15}

## Define metrics (LLM as a judge) 

In [74]:
from langchain_groq import ChatGroq
from langsmith import wrappers

eval_instructions = """You are an evaluator assessing the quality of a chatbot response for the Helwan Chemical Industries Company website.

The chatbot is intended to provide accurate, professional, and grounded information about the company, its products, operations, and policies. All answers should reflect information that would reasonably be found on the official Helwan Chemical Industries Company website or standard public company communications.

Evaluate the chatbot response according to the following criteria:

Correctness:
Determine whether the response correctly answers the user's question. The information should be factually accurate and free from errors or misleading claims.

Groundedness:
Check whether the response is grounded in realistic, verifiable company information. The chatbot must not invent details such as certifications, export countries, production capacities, financial figures, or internal processes unless such information is clearly public. Conservative answers are preferred over speculative ones.

Completeness:
Assess whether the response adequately covers the user's question. The answer should include all essential points without being unnecessarily long or overly brief.

Clarity and Tone:
The response should be clear, easy to understand, and written in a professional tone appropriate for an industrial and corporate website.

Safety and Compliance:
Ensure the response does not provide unsafe chemical guidance, operational instructions, legal advice, or environmental claims beyond high-level compliance statements.

If the user asks a question that cannot be confidently answered based on publicly available company information, the correct behavior is for the chatbot to clearly state that the information is not available and suggest contacting the company directly. The chatbot must not guess or hallucinate an answer.

After reviewing the response, provide a final judgment indicating whether the response passes or fails the evaluation, along with a brief explanation for the decision."""

default_instruction = "Responde to the users question in a short, concise manner (one short sentence)."

In [75]:
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""
    You are grading the following question:
    {inputs['input']}

    Here is the correct answer:
    {reference_outputs['output']}

    You are grading the following predicted answer:
    {outputs['response']}

    Respond with only one word: CORRECT or INCORRECT.
    Grade:
    """

    # Create the ChatGroq instance
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",  # Use a valid model name
        temperature=0
    )
    
    # Invoke the model with messages
    response = llm.invoke([
        {"role": "system", "content": eval_instructions},
        {"role": "user", "content": user_content}
    ])
    
    # Extract the text content from the response
    grade = response.content.strip().upper()
    
    return grade == "CORRECT"

In [76]:
# Concision evaluator
def concision(outputs: dict, reference_outputs: dict) -> bool:
    # Get the actual string responses
    predicted = outputs.get('response', '')
    reference = reference_outputs.get('output', '')
    
    # Convert to string if needed
    if not isinstance(predicted, str):
        predicted = str(predicted)
    if not isinstance(reference, str):
        reference = str(reference)
    
    # Check if prediction is less than 2x the reference length
    return int(len(predicted) < 2 * len(reference))

## Run Evaluation

In [77]:
default_instruction = "Responde to the users question in a short, concise manner (one short sentence)."

In [78]:
def my_app(question: str, model: str = "llama-3.3-70b-versatile", instructions: str = default_instruction) -> str:
    # Create the LLM instance
    llm = ChatGroq(
        model=model,
        temperature=0
    )
    
    # Invoke and get the response
    response = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ])
    
    # Return the actual text content
    return response.content

In [79]:
# Call my_app for every datapoint
def ls_target(inputs: dict) -> dict:
    try:
        return {
            "response": my_app(inputs["input"])
        }
    except Exception as e:
        print(f"Error processing input: {e}")
        return {
            "response": ""
        }

In [80]:
experiment_results = client.evaluate(
    ls_target,
    data="Helwan Chem chatbot evaluation",
    evaluators=[correctness, concision],
    experiment_prefix="helwanChem"
)

View the evaluation results for experiment: 'helwanChem-3060ac12' at:
https://smith.langchain.com/o/45b74521-3dcf-4e51-998a-7aa2125ec7a2/datasets/33cacf2a-685b-4771-ad8e-8de9f54918e7/compare?selectedSessions=a3863ccf-9d0a-4f8d-b882-ed72a8287e1c




15it [00:15,  1.04s/it]
